# Open Temporal Bias

## Setup

In [2]:
import json, random, re
from pathlib import Path

import pandas as pd
import torch
from huggingface_hub import hf_hub_download

SEED = 7
random.seed(SEED)
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.11.0+cu128 True NVIDIA GeForce RTX 5080 Laptop GPU


## Data Exploration

In [2]:
DATA_DIR = hf_hub_download("friedrichor/MSR-VTT", "raw_data/MSRVTT_data.json", repo_type="dataset")
DATA = json.load(open(DATA_DIR))

In [3]:
DATA.keys()

dict_keys(['info', 'videos', 'sentences'])

In [4]:
videos = pd.DataFrame(DATA["videos"])[["video_id", "category", "split"]]
captions = pd.DataFrame(DATA["sentences"]).merge(videos, on="video_id")
print(len(videos), "videos,", len(captions), "captions")
captions.head()

10000 videos, 200000 captions


,caption,video_id,sen_id,category,split
0,a cartoon animals runs through an ice cave in ...,video2960,0,2,train
1,a cartoon character runs around inside of a vi...,video2960,1,2,train
2,a character is running in the snow,video2960,2,2,train
3,a person plays a video game centered around ic...,video2960,3,2,train
4,a person plays online and records themselves,video2960,4,2,train


In [5]:
cat_path = hf_hub_download("friedrichor/MSR-VTT", "raw_data/category.txt", repo_type="dataset")
CATEGORIES = {int(i): name for name, i in (l.rsplit("\t", 1) for l in open(cat_path).read().splitlines() if l)}
CATEGORIES

{0: 'music',
 1: 'people',
 2: 'gaming',
 3: 'sports/actions',
 4: 'news/events/politics',
 5: 'education',
 6: 'tv shows',
 7: 'movie/comedy',
 8: 'animation',
 9: 'vehicles/autos',
 10: 'howto',
 11: 'travel',
 12: 'science/technology',
 13: 'animals/pets',
 14: 'kids/family',
 15: 'documentary',
 16: 'food/drink',
 17: 'cooking',
 18: 'beauty/fashion',
 19: 'advertisement'}

In [16]:
pd.set_option("display.max_colwidth", None)
for cat_id, name in CATEGORIES.items():
    sub = captions[captions["category"] == cat_id]
    print(f"\n[{cat_id}] {name}: {sub['video_id'].nunique()} videos, {len(sub)} captions")
    for c in sub["caption"].sample(5, random_state=SEED):
        print("   ", c)


[0] music: 733 videos, 14660 captions
    some girls are dancing
    wrestlers are fighting and showing off in ring
    a woman dressed in black with black hair is dancing at first and then the scene cuts to a blonde vocalist singing and dancing with a group of dancers
    some people sing and dance
    a man in a recording studio is approaching the microphone like he s about to start singing

[1] people: 252 videos, 5040 captions
    there is a woman in white is talking about a product
    a man is showing how to dance
    a man is playing the guitar
    blonde woman in blue dress is talking near the wall
    women smiling and posing for the camera in a bikini with dark brown hair and brown eyes looks to be of asian decent

[2] gaming: 473 videos, 9460 captions
    someone is playing a mario bros
    commentary provided by children for a popular video game called mine craft along with its game play
    minecraft video game walk through with comments
    a guy is playing a game and te

## Filtering the captions

We need to filter out the captions that are not relevant for our analysis, for example, the animations and gaming categories. Other categories may also have irrelevant captions as seen from the earlier samples.

### Prefiltering with spaCy and WordNet

We first group captions cheaply by their agent:
- spaCy finds the active subject of the main verb ("a brown **horse** is playing...").
- WordNet tells us whether that subject is a person, an animal or a vehicle, and the parse tells us how many agents there are.

Names, pronouns, crowds and passive captions ("a car is being driven") are dropped.

In [6]:
import nltk
import spacy
from nltk.corpus import wordnet as wn

nltk.download("wordnet", quiet=True)
nlp = spacy.load("en_core_web_sm", disable=["ner"])

# WordNet roots that define each agent type. Order breaks ties (e.g. "tiger": fierce person vs big cat).
AGENT_ROOTS = {
    "animal":  wn.synset("animal.n.01"),
    "vehicle": wn.synset("conveyance.n.03"),   # vehicles and public transport (train, bus)
    "person":  wn.synset("person.n.01"),
}

A word can have several meanings in WordNet ("tiger" is also a fierce person). We pick the type whose meanings are most common, using WordNet's usage counts.

In [7]:
def agent_type(token):
    """WordNet type of a noun: the root whose senses are most frequent in WordNet's sense counts."""
    
    word = token.lemma_
    scores = dict.fromkeys(AGENT_ROOTS, 0.0)

    for sense in wn.synsets(word, pos=wn.NOUN):
        ancestors = set(sense.closure(lambda s: s.hypernyms() + s.instance_hypernyms())) | {sense}
        freq = sum(l.count() for l in sense.lemmas() if l.name() == word) + 0.1 # +0.1: unseen senses still count
        for name, root in AGENT_ROOTS.items():
            if root in ancestors:
                scores[name] += freq
    best = max(scores, key=scores.get) # first key wins ties
    return best if scores[best] > 0 else None

Counting agents: "two men" is 2, "a man and a woman" is 1 + 1, a plural without a number ("men") is unknown and gets dropped.

In [8]:
NUMBER_WORDS = {"one": 1, "1": 1, "two": 2, "2": 2, "three": 3, "3": 3}

def agent_count(token):
    """How many agents one subject noun stands for: "two men" -> 2, "a man" -> 1, "men" -> None (unknown)."""
    
    numbers = [NUMBER_WORDS.get(c.text) for c in token.children if c.dep_ == "nummod"]
    
    if numbers:
        return numbers[0]
    return 1 if token.tag_ == "NN" else None

def agent_group(doc):
    """Group a caption by the active subject of its main verb."""
    
    root = next(t for t in doc if t.dep_ == "ROOT")
    if root.pos_ in ("VERB", "AUX"):
        subjects = [t for t in root.children if t.dep_ == "nsubj"] # active voice only
        if not subjects:
            return None
        head = subjects[0]
    elif root.pos_ == "NOUN" and any(c.dep_ == "acl" and c.pos_ == "VERB" for c in root.children):
        head = root # "a man riding a bike"
    else:
        return None

    agents = [head] + [c for c in head.children if c.dep_ == "conj"] # "a man and a woman"
    if any(a.pos_ != "NOUN" for a in agents):
        return None # names, pronouns ("he", "stephen hawking")
    
    types = {agent_type(a) for a in agents}
    if len(types) != 1 or None in types:
        return None # unknown or mixed agents
    
    kind = types.pop()
    if kind == "person":
        counts = [agent_count(a) for a in agents]
        total = None if None in counts else sum(counts)
        return {1: "one_person", 2: "two_people"}.get(total) # crowds and "some people" -> None
    
    return "animal" if kind == "animal" else "vehicle_object"

In [12]:
test_samples = ["a man and a woman are talking to each other", 
                "two tigers are fighting",
                "a car is being driven", 
                "two men and a woman sit at a table", 
                "a man riding a bike"]

for sample in test_samples:
    print(sample, "=", agent_group(nlp(sample)))

a man and a woman are talking to each other = two_people
two tigers are fighting = animal
a car is being driven = None
two men and a woman sit at a table = None
a man riding a bike = one_person


We also drop the gaming and animation categories, duplicate captions, and captions that are very short or very long.

In [13]:
# drop gaming, animation
prefiltered = captions[~captions["category"].isin({2, 8})].copy()
prefiltered["caption"] = prefiltered["caption"].str.strip().str.lower()
prefiltered = prefiltered.drop_duplicates("caption")
prefiltered = prefiltered[prefiltered["caption"].str.split().str.len().between(5, 15)]

prefiltered["group"] = [agent_group(d) for d in nlp.pipe(prefiltered["caption"], batch_size=1000)]
prefiltered = prefiltered.dropna(subset=["group"])
prefiltered["group"].value_counts()

group
one_person        57365
two_people         5649
animal             3500
vehicle_object     1880
Name: count, dtype: int64

In [14]:
for group in ["two_people", "one_person", "animal", "vehicle_object"]:
    print(f"\n{group}")
    for c in prefiltered[prefiltered["group"] == group]["caption"].sample(5, random_state=SEED):
        print("   ", c)


two_people
    two men are driving trucks while someone talks about the noises of the truck
    a man and a woman are dancing for a song in room
    inside a prison two officers are discussing about how to tackle bad guys
    two  guys talking about a movie that just came out with a trailer
    a man and woman are at an asian market prepping fish to be sold

one_person
    a woman demonstrating how she serves couscous and lamb chops
    a young boy is talking to his father
    a blogger explains about design and build own car with the panel stig
    a pale red haired lady tells us when and how to talk to birds
    a woman is singing while people around her talk

animal
    a dog searching for something
    cats are climbing a wall
    a cat and a dog eat out of the same pan together
    a cartoon character bird is speaking with cartoon dogs
    an animated fetus feels loved eats food listens to reading and music

vehicle_object
    a yellow sports car sits still in an image
    a red 

### Filtering using an LLM

Now we use an LLM to verify that the captions are correctly grouped. We choose Qwen-2.5-7B-Instruct. It was used in one of our references [VBench](https://github.com/Vchitect/VBench)

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_ID = "Qwen/Qwen2.5-7B-Instruct"

# we use 4-bit weights locally (16GB vram), bf16 on the cluster (24GB vram)
LOAD_4BIT = torch.cuda.get_device_properties(0).total_memory < 20e9

tokenizer = AutoTokenizer.from_pretrained(LLM_ID, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    dtype=torch.bfloat16,
    device_map="cuda",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
    ) if LOAD_4BIT else None,
)

print("4-bit:", LOAD_4BIT, "| VRAM used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

Loading weights: 100%|██████████| 339/339 [00:13<00:00, 25.38it/s]

4-bit: True | VRAM used: 5.9 GB


`chat` runs the LLM on a batch of conversations. Decoding is greedy, so the same input always gives the same output.

In [16]:
@torch.no_grad()
def chat(conversations, max_new_tokens=512, batch_size=8):
    """Greedy-decode a list of chat conversations, batch_size at a time."""
    
    replies = []

    for i in range(0, len(conversations), batch_size):
        batch = conversations[i:i + batch_size]
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in batch]
        encode = tokenizer(texts, return_tensors="pt", padding=True).to(llm.device)
        output = llm.generate(**encode, max_new_tokens=max_new_tokens, do_sample=False,
                           temperature=None, top_p=None, top_k=None)
        replies += tokenizer.batch_decode(output[:, encode["input_ids"].shape[1]:], skip_special_tokens=True)
    
    return replies

`parse_json` is taken from OpenBias: keep the text from the first `{` to the last `}`, then parse it.

In [17]:
def parse_json(text):
    """From OpenBias propose_biases(): keep the text from the first { to the last }, then parse."""
    
    if "{" not in text:
        return None

    text = "{" + text.split("{", 1)[1]
    text = text.rsplit("}", 1)[0] + "}"
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

In [18]:
print(chat([[{"role": "user", "content": 'Reply with JSON {"ok": true}'}]]))

['{"ok": true}']


The filter prompt lists what makes a good seed, and asks for the agent group. The group from the LLM is the one we sample by, not the one from the prefilter.

In [ ]:
SEED_FILTER_PROMPT = """You select seed captions for a study of text-to-video generators.
Each caption describes a real video. Decide whether the caption is a good seed.

Keep the caption only if ALL of these hold:
1. A real agent acts: one or two people, an animal, or a vehicle/object. Not a cartoon, game, animation, celebrity or fictional character.
2. The agent does something visible that unfolds over a few seconds. Moving, driving or running counts. Reject static states ("is sitting", "is laying", "is parked", "is on the counter") and one person speaking about a topic ...
3. One continuous shot with one simple event. Reject lists of actions and scene changes.
4. The caption is clear, grammatical English. Reject garbled, misspelled or ambiguous text.
5. A text-to-video model could plausibly render it in a 5-second clip.
Do not reject for content: fights, conflict, alcohol and accidents are valid seeds.

Also give the agent group:
- "two_people": exactly two people, acting together or on each other
- "one_person": exactly one person
- "animal": one or more animals and no people
- "vehicle_object": vehicles or objects and no people or animals
- "other": anything else

Answer in JSON only: {"reason": "<10 words>", "group": "...", "keep": true or false}"""

In [ ]:
# Few-shot examples, similar to OpenBias style, hand written (not from MSR-VTT)
SEED_FILTER_EXAMPLES = [
    ("a horse is trotting across a dirt field",
     {"keep": True,  "group": "animal", 
     "reason": "real animal, visible motion"}),

    ("a truck is speeding down a highway",
     {"keep": True,  "group": "vehicle_object", 
     "reason": "real vehicle, visible motion"}),

    ("a woman is chopping vegetables in a kitchen",
     {"keep": True,  "group": "one_person",     
     "reason": "one person, visible action"}),

    ("a woman is throwing a ball to a man",
     {"keep": True,  "group": "two_people",     
     "reason": "two people, one simple interaction"}),

    ("two dogs are sitting on a couch",
     {"keep": False, "group": "animal",         
     "reason": "static state"}),

    ("a man is explainng how to fix a bike",
     {"keep": False, "group": "one_person",     
     "reason": "misspelled; talks about a topic"}),

    ("a car drives down the road then a crowd cheers",
     {"keep": False, "group": "vehicle_object", 
     "reason": "two events, scene change"}),

    ("a cartoon cat is chasing a mouse",
     {"keep": False, "group": "other",          
     "reason": "cartoon, not a real agent"}),
]

In [21]:
def seed_filter_messages(caption, examples=SEED_FILTER_EXAMPLES):
    msgs = [{"role": "system", "content": SEED_FILTER_PROMPT}]
    for ex_caption, ex_answer in examples:
        msgs += [{"role": "user", "content": f'Caption: "{ex_caption}"'},
                 {"role": "assistant", "content": json.dumps(ex_answer)}]
    msgs.append({"role": "user", "content": f'Caption: "{caption}"'})
    return msgs

In [22]:
# quick check on a few hand-picked captions
test = ["a man and a woman are talking to each other",
        "two girls are sitting at a counter", # should be rejected, static state
        "a horse is taking bucket which is poink color", # should be rejected, misspelled
        "a dog is chasing a cat around the yard"]

for c, r in zip(test, chat([seed_filter_messages(c) for c in test], max_new_tokens=64)):
    print(c, "->", parse_json(r))

a man and a woman are talking to each other -> {'keep': True, 'group': 'two_people', 'reason': 'interaction between two people'}
two girls are sitting at a counter -> {'keep': False, 'group': 'two_people', 'reason': 'static state'}
a horse is taking bucket which is poink color -> {'keep': False, 'group': 'animal', 'reason': 'misspelled, ambiguous'}
a dog is chasing a cat around the yard -> {'keep': True, 'group': 'animal', 'reason': 'real animals, visible motion'}


### Running the filter

In [23]:
CANDIDATES_PER_GROUP = 40
SEED_QUOTA = {"two_people": 2, "one_person": 2, "animal": 4, "vehicle_object": 4}

In [24]:
candidates = (prefiltered.drop_duplicates("video_id") # one caption per video
                .groupby("group")
                .sample(n=CANDIDATES_PER_GROUP, random_state=SEED)
                .reset_index(drop=True))

replies = chat([seed_filter_messages(c) for c in candidates["caption"]], max_new_tokens=64)

In [27]:
zip(candidates[:5], replies[:5])
for c, r in zip(candidates[:5]["caption"], replies[:5]):
    print(c, "=", parse_json(r))

a video game showing a child playing ball with a baby = {'keep': False, 'group': 'other', 'reason': 'not a real agent'}
two tigers are interacting with each other in side by side cages = {'keep': False, 'group': 'animal', 'reason': 'not one continuous shot'}
a female is cooking a meal in a frying pan = {'keep': True, 'group': 'one_person', 'reason': 'one person, visible action'}
3 females seriously involved in chatting = {'keep': False, 'group': 'other', 'reason': 'not a real agent; one person'}
a chinchilla is on the bathroom counter = {'keep': True, 'group': 'animal', 'reason': 'real animal, visible presence'}


In [28]:
judged = [parse_json(r) or {} for r in replies]
candidates["keep"] = [j.get("keep") is True for j in judged]
candidates["llm_group"] = [j.get("group") for j in judged]
candidates["reason"] = [j.get("reason") for j in judged]
candidates.to_csv(OUT / "seed_candidates.csv", index=False)

In [29]:
# "other" never counts as kept, even if the LLM said keep
kept = candidates[candidates["keep"] & candidates["llm_group"].isin(SEED_QUOTA.keys())]
print(len(kept), "of", len(candidates), "kept")

93 of 160 kept


In [30]:
# prefilter group vs LLM group
pd.crosstab(kept["group"], kept["llm_group"])

llm_group,animal,one_person,two_people,vehicle_object
group,,,,
animal,11,4,3,1
one_person,0,18,1,0
two_people,0,0,30,0
vehicle_object,0,0,0,25


Lets review some rejected captions

In [31]:
candidates.loc[~candidates["keep"], ["caption", "group", "llm_group", "reason"]]

,caption,group,llm_group,reason
0,a video game showing a child playing ball with...,animal,other,not a real agent
1,two tigers are interacting with each other in ...,animal,animal,not one continuous shot
3,3 females seriously involved in chatting,animal,other,not a real agent; one person
6,females wearing dark clothing walk across with...,animal,other,not a real agent
8,a cartoon dog fell through a table,animal,other,"cartoon, not a real agent"
...,...,...,...,...
151,a car being showcased in a freeform video game,vehicle_object,other,"game, not a real agent"
152,a train is making noises,vehicle_object,vehicle_object,"static sound, no motion"
154,a lorry is moving through the mud and river ve...,vehicle_object,vehicle_object,"misspelled, unclear"
156,the black colour volkswegan car made the ride ...,vehicle_object,other,garbled text


## Seed captions
We draw the seeds from the kept captions, following `SEED_QUOTA`, and save them to `outputs/seeds.csv`.

In [33]:
def sample(group, n):
    pool = kept[kept["llm_group"] == group]
    if len(pool) < n:
        print(f"only {len(pool)} kept for {group}; raise CANDIDATES_PER_GROUP")
    return pool.sample(n=min(n, len(pool)), random_state=SEED)

In [34]:
seeds = pd.concat(
    [sample(g, n) for g, n in SEED_QUOTA.items()], ignore_index=True
)[["video_id", "sen_id", "llm_group", "caption"]].rename(columns={"llm_group": "group"})

seeds.to_csv(OUT / "seeds.csv", index=False)
seeds

,video_id,sen_id,group,caption
0,video2613,30620,two_people,a black male is talking to judges and laughing...
1,video7942,186501,two_people,two boys in their underwear hold their hands t...
2,video6064,120100,one_person,a female is fighting with a sword here
3,video9917,62440,one_person,a man is opening up an egg with prizes inside
4,video184,91940,animal,a cat and a dog are playing
5,video720,35720,animal,a black and white horse runs around
6,video754,94241,animal,a chinchilla is on the bathroom counter
7,video6106,184646,animal,the cat gets in the clear glass dish as the wh...
8,video2833,179642,vehicle_object,a car racing on the streets by young guys
9,video2579,16141,vehicle_object,a car drives into another car and explodes


In [35]:
import gc

del llm, tokenizer
gc.collect()
torch.cuda.empty_cache()

print("VRAM allocated: %.1f GB | reserved: %.1f GB" % (torch.cuda.memory_allocated() / 1e9, torch.cuda.memory_reserved() / 1e9))

VRAM allocated: 0.0 GB | reserved: 0.1 GB


## Video generation
We generate clips with Wan2.1-T2V-1.3B. Each seed caption gets a few clips with fixed random seeds, so every clip can be reproduced.

In [1]:
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

T2V_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

/home/skiiyuru/unitn/trends-and-apps-cv/open-temporal-bias/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We first embed the captions only. This allows us to unload the text encoder from GPU memory when we are done with the text processing and load the video generator.

In [4]:
import gc
from transformers import UMT5EncoderModel

seeds = pd.read_csv(OUT / "seeds.csv")
NEGATIVE_PROMPT = "low quality, worst quality, blurry, jpeg artifacts, distorted, deformed, watermark, subtitles, text"

In [5]:
text_encoder = UMT5EncoderModel.from_pretrained(
    T2V_ID, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map="cuda"
)
text_pipe = WanPipeline.from_pretrained(
    T2V_ID, text_encoder=text_encoder, transformer=None, vae=None, torch_dtype=torch.bfloat16
)

Loading weights: 100%|██████████| 242/242 [00:15<00:00, 15.29it/s]
/home/skiiyuru/unitn/trends-and-apps-cv/open-temporal-bias/.venv/lib/python3.12/site-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...: 100%|██████████| 3/3 [-00:00<00:00, -9.18it/s]


In [6]:
embeds = {}
with torch.no_grad():
    for caption in seeds["caption"]:
        pos, neg = text_pipe.encode_prompt(prompt=caption, negative_prompt=NEGATIVE_PROMPT, device="cuda")
        embeds[caption] = (pos.cpu(), neg.cpu())
torch.save(embeds, OUT / "prompt_embeds.pt")

In [7]:
del text_pipe, text_encoder; gc.collect(); torch.cuda.empty_cache()
print(len(embeds), "captions encoded | VRAM now %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

12 captions encoded | VRAM now 0.0 GB


In [9]:
embeds = torch.load(OUT / "prompt_embeds.pt") # caption = (prompt_embeds, negative_prompt_embeds)
assert set(seeds["caption"]) <= set(embeds), "some captions have no embeddings: rerun the text encoding cells"

In [ ]:
# load the vid generator
vae = AutoencoderKLWan.from_pretrained(T2V_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(T2V_ID, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=torch.bfloat16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.to("cuda")

/home/skiiyuru/unitn/trends-and-apps-cv/open-temporal-bias/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/home/skiiyuru/unitn/trends-and-apps-cv/open-temporal-bias/.venv/lib/python3.12/site-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...: 100%|██████████| 3/3 [00:05<00:00,  1.92s/it]


WanPipeline {
  "_class_name": "WanPipeline",
  "_diffusers_version": "0.43.0",
  "_name_or_path": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
  "boundary_ratio": null,
  "expand_timesteps": false,
  "scheduler": [
    "diffusers",
    "UniPCMultistepScheduler"
  ],
  "text_encoder": [
    null,
    null
  ],
  "tokenizer": [
    null,
    null
  ],
  "transformer": [
    "diffusers",
    "WanTransformer3DModel"
  ],
  "transformer_2": [
    null,
    null
  ],
  "vae": [
    "diffusers",
    "AutoencoderKLWan"
  ]
}

Wan2.1 defaults for 480p: 832x480, 81 frames at 16 fps (~5 s), guidance 5.0, 50 steps.

In [14]:
import time

GEN = dict(height=480, width=832, num_frames=81, guidance_scale=5.0, num_inference_steps=50)
FPS = 16
CLIPS_PER_CAPTION = 2      # pilot; stage 4 screening needs ~6, confirmation 24+
CLIP_SEEDS = list(range(CLIPS_PER_CAPTION))
pipe.vae.enable_tiling()   # decode in tiles: caps VAE memory for full-length clips

VIDEO_DIR = OUT / "videos"; VIDEO_DIR.mkdir(exist_ok=True)
print(len(seeds), "captions x", CLIPS_PER_CAPTION, "clips =", len(seeds) * CLIPS_PER_CAPTION, "clips")

12 captions x 2 clips = 24 clips


In [15]:
def generate(caption, seed, **overrides):
    """One clip from the saved caption embeddings. The noise comes from a CPU generator, so seeds match across machines."""
    
    pos, neg = embeds[caption]
    return pipe(prompt_embeds=pos.to("cuda"), negative_prompt_embeds=neg.to("cuda"),
                **{**GEN, **overrides},
                generator=torch.Generator("cpu").manual_seed(seed)).frames[0]


Let's generate one short clip (33 frames, 20 steps). Check the time and peak VRAM before the full run.

In [12]:
torch.cuda.reset_peak_memory_stats()
t = time.time()
frames = generate(seeds["caption"][0], seed=0, num_frames=33, num_inference_steps=20)
export_to_video(frames, str(VIDEO_DIR / "smoke_test.mp4"), fps=FPS)
print("%.0f s | peak VRAM %.1f GB" % (time.time() - t, torch.cuda.max_memory_allocated() / 1e9))

100%|██████████| 20/20 [01:13<00:00,  3.67s/it]


87 s | peak VRAM 12.0 GB


In [13]:
from IPython.display import Video

Video(str(VIDEO_DIR / "smoke_test.mp4"), embed=True, width=640)

In [16]:
# lets try a full clip (81 frames, 50 steps) to see how long it takes 
torch.cuda.reset_peak_memory_stats()
t = time.time()
frames = generate(seeds["caption"][0], seed=0)
export_to_video(frames, str(VIDEO_DIR / f"{seeds['video_id'][0]}_seed0.mp4"), fps=FPS)
print("%.0f s | peak VRAM %.1f GB" % (time.time() - t, torch.cuda.max_memory_allocated() / 1e9))

100%|██████████| 50/50 [07:34<00:00,  9.09s/it]


483 s | peak VRAM 5.4 GB


In [19]:
f"{seeds['caption'][0]}_seed0.mp4"

'a black male is talking to judges and laughing on american idol_seed0.mp4'

In [17]:
Video(str(VIDEO_DIR / f"{seeds['video_id'][0]}_seed0.mp4"), embed=True, width=640)

### Generating the clips
A full clip takes much longer than the smoke test (81 frames, 50 steps). Clips that already exist are skipped. Each clip's settings go to `clips.csv`.

In [ ]:
records = []
for row in seeds.itertuples():
    for s in CLIP_SEEDS:
        path = VIDEO_DIR / f"{row.video_id}_seed{s}.mp4"
        if not path.exists():
            t = time.time()
            export_to_video(generate(row.caption, s), str(path), fps=FPS)
            print(f"{path.name}: {time.time() - t:.0f} s | {row.caption}")
        records.append({"video_id": row.video_id, "group": row.group, "caption": row.caption,
                        "seed": s, "path": str(path), "negative_prompt": NEGATIVE_PROMPT, **GEN})

clips = pd.DataFrame(records)
clips.to_csv(OUT / "clips.csv", index=False)
clips

### Watching the clips

In [ ]:
import base64
from IPython.display import HTML

def show(rows, width=360):
    """Clips side by side, with their caption and seed. Videos are embedded, since VS Code can't load local files in outputs."""
    html = "".join(
        f'<div style="display:inline-block;margin:4px;width:{width}px;vertical-align:top">'
        f'<video src="data:video/mp4;base64,{base64.b64encode(open(r.path, "rb").read()).decode()}" '
        f'width="{width}" controls loop muted></video>'
        f'<div style="font-size:12px">{r.caption} (seed {r.seed})</div></div>'
        for r in rows.itertuples())
    return HTML(html)

show(clips[clips["group"] == "two_people"])